# Module 8 Lab — Human Oversight & Bounded Autonomy

**Scenario:** An enterprise Procurement Agent operates with progressively increasing autonomy.

We will build the control layer around the agent rather than asking the model whether it needs supervision.

In [ ]:
%pip install -q "pydantic>=2" pandas
print("Dependencies installed.")

In [ ]:
from __future__ import annotations
from pydantic import BaseModel, Field
from enum import Enum
from typing import Any, Optional
from datetime import datetime, timezone, timedelta
from uuid import uuid4
import hashlib, json, copy
import pandas as pd
pd.set_option("display.max_colwidth",120)

## 1. Define an autonomy envelope

In [ ]:
class AutonomyEnvelope(BaseModel):
    agent:str
    allowed_tools:set[str]
    autonomous_amount:float
    task_spend_limit:float
    max_risk_score:float
    allow_irreversible:bool=False
    max_delegation_depth:int=1
    valid_until:datetime

ENVELOPE=AutonomyEnvelope(
    agent="agent:procurement-v3",
    allowed_tools={"vendor.lookup","po.create","po.cancel"},
    autonomous_amount=5000,
    task_spend_limit=10000,
    max_risk_score=.55,
    allow_irreversible=False,
    max_delegation_depth=1,
    valid_until=datetime.now(timezone.utc)+timedelta(days=90)
)
ENVELOPE

## 2. Normalize the proposed action

In [ ]:
class Action(BaseModel):
    action_id:str=Field(default_factory=lambda:f"act-{uuid4().hex[:8]}")
    subject:str
    agent:str
    task_id:str
    tool:str
    resource:str
    amount:float=0
    vendor_id:Optional[str]=None
    reversible:bool=True
    data_sensitivity:int=1
    novelty:float=0
    anomaly_score:float=0
    delegation_depth:int=0

a=Action(subject="user:123",agent=ENVELOPE.agent,task_id="task:123",
         tool="po.create",resource="department:data-ai",amount=4500,
         vendor_id="vendor-acme",reversible=True,novelty=.1,anomaly_score=.1)
a

## 3. Multi-factor risk score

In [ ]:
def risk_score(a:Action)->float:
    impact=min(a.amount/50000,1.0)
    irreversibility=0 if a.reversible else 1
    sensitivity=min(a.data_sensitivity/4,1)
    delegation=min(a.delegation_depth/3,1)
    score=(.35*impact + .20*irreversibility + .15*a.novelty +
           .15*a.anomaly_score + .10*sensitivity + .05*delegation)
    return round(min(score,1),3)

for amount in [40,2000,25000,500000]:
    x=a.model_copy(update={"amount":amount})
    print(amount,risk_score(x))

## 4. Route oversight by risk and policy

In [ ]:
class Route(str,Enum):
    AUTO_ALLOW="AUTO_ALLOW"
    VERIFY="VERIFY"
    APPROVE="APPROVE"
    MULTI_APPROVE="MULTI_APPROVE"
    DENY="DENY"

def route(a:Action,e:AutonomyEnvelope)->tuple[Route,str]:
    if datetime.now(timezone.utc)>e.valid_until:
        return Route.DENY,"Autonomy grant expired."
    if a.agent!=e.agent or a.tool not in e.allowed_tools:
        return Route.DENY,"Outside autonomy envelope."
    if not a.reversible and not e.allow_irreversible:
        return Route.MULTI_APPROVE,"Irreversible action."
    if a.delegation_depth>e.max_delegation_depth:
        return Route.DENY,"Delegation depth exceeded."
    r=risk_score(a)
    if a.amount>25000 or r>=.75:
        return Route.MULTI_APPROVE,"Critical consequence."
    if a.amount>e.autonomous_amount or r>e.max_risk_score:
        return Route.APPROVE,"Outside autonomous threshold."
    if a.novelty>.5 or a.anomaly_score>.5:
        return Route.VERIFY,"Additional verification required."
    return Route.AUTO_ALLOW,"Inside bounded autonomy envelope."

rows=[]
for amount in [40,2000,5001,25000,500000]:
    x=a.model_copy(update={"amount":amount})
    rt,reason=route(x,ENVELOPE)
    rows.append({"amount":amount,"risk":risk_score(x),"route":rt.value,"reason":reason})
display(pd.DataFrame(rows))

## 5. Approval request: show the actual action

In [ ]:
def canonical_action(a:Action)->dict:
    return a.model_dump(mode="json")

def digest_action(a:Action)->str:
    raw=json.dumps(canonical_action(a),sort_keys=True,separators=(",",":"))
    return hashlib.sha256(raw.encode()).hexdigest()

class ApprovalRequest(BaseModel):
    approval_id:str=Field(default_factory=lambda:f"apr-{uuid4().hex[:8]}")
    action_id:str
    action_digest:str
    summary:dict[str,Any]
    required_roles:set[str]
    quorum:int=1
    created_at:datetime=Field(default_factory=lambda:datetime.now(timezone.utc))
    expires_at:datetime=Field(default_factory=lambda:datetime.now(timezone.utc)+timedelta(hours=4))

def make_approval(a:Action,multi=False):
    return ApprovalRequest(
        action_id=a.action_id,
        action_digest=digest_action(a),
        summary={
            "tool":a.tool,"vendor":a.vendor_id,"amount":a.amount,
            "task":a.task_id,"agent":a.agent,"risk":risk_score(a),
            "reversible":a.reversible
        },
        required_roles={"manager","finance"} if multi else {"manager"},
        quorum=2 if multi else 1
    )

high=a.model_copy(update={"amount":9000})
approval=make_approval(high)
approval

## 6. Approval integrity: prevent post-approval mutation

In [ ]:
tampered=high.model_copy(update={"amount":19000})
print("approved digest :",approval.action_digest)
print("current digest  :",digest_action(high))
print("tampered digest :",digest_action(tampered))
print("approved action unchanged:",approval.action_digest==digest_action(high))
print("tampered accepted:",approval.action_digest==digest_action(tampered))

## 7. Reviewer authority + separation of duties

In [ ]:
REVIEWERS={
 "manager:456":{"roles":{"manager"},"user":"user:456"},
 "finance:777":{"roles":{"finance"},"user":"user:777"},
 "user:123":{"roles":{"manager"},"user":"user:123"} # requester
}

class ApprovalDecision(BaseModel):
    reviewer:str
    approved:bool
    timestamp:datetime=Field(default_factory=lambda:datetime.now(timezone.utc))

def reviewer_can_decide(req:ApprovalRequest,a:Action,reviewer:str)->tuple[bool,str]:
    if reviewer not in REVIEWERS: return False,"Unknown reviewer."
    if REVIEWERS[reviewer]["user"]==a.subject:
        return False,"Separation-of-duties violation: requester cannot approve own action."
    if not (REVIEWERS[reviewer]["roles"] & req.required_roles):
        return False,"Reviewer lacks required role."
    if datetime.now(timezone.utc)>req.expires_at:
        return False,"Approval request expired."
    return True,"Authorized reviewer."

for r in REVIEWERS:
    print(r,reviewer_can_decide(approval,high,r))

## 8. Quorum and multi-party approval

In [ ]:
critical=a.model_copy(update={"amount":50000,"reversible":False})
multi=make_approval(critical,multi=True)
decisions=[
    ApprovalDecision(reviewer="manager:456",approved=True),
    ApprovalDecision(reviewer="finance:777",approved=True)
]

def approval_status(req:ApprovalRequest,a:Action,decisions:list[ApprovalDecision]):
    valid=[]
    for d in decisions:
        ok,_=reviewer_can_decide(req,a,d.reviewer)
        if ok: valid.append(d)
    if any(not d.approved for d in valid): return "REJECTED"
    distinct_roles=set()
    for d in valid:
        if d.approved:
            distinct_roles |= (REVIEWERS[d.reviewer]["roles"] & req.required_roles)
    return "APPROVED" if len(distinct_roles)>=req.quorum else "PENDING"

approval_status(multi,critical,decisions)

## 9. Durable pause/resume state

In [ ]:
class WorkflowState(BaseModel):
    workflow_id:str=Field(default_factory=lambda:f"wf-{uuid4().hex[:8]}")
    action:Action
    status:str="RUNNING"
    pending_approval:Optional[ApprovalRequest]=None
    policy_version:str="v1"
    revoked:bool=False

def checkpoint_for_approval(a:Action)->WorkflowState:
    rt,_=route(a,ENVELOPE)
    req=make_approval(a,multi=(rt==Route.MULTI_APPROVE))
    return WorkflowState(action=a,status="WAITING_FOR_APPROVAL",pending_approval=req)

state=checkpoint_for_approval(high)
serialized=state.model_dump_json()
print(serialized[:500]+"...")
restored=WorkflowState.model_validate_json(serialized)
print(restored.status,restored.workflow_id)

## 10. Revalidate before execution

In [ ]:
SANCTIONED=set()

def pre_execute(state:WorkflowState,decisions:list[ApprovalDecision])->tuple[bool,str]:
    if state.revoked: return False,"Authority revoked."
    a=state.action
    if a.vendor_id in SANCTIONED: return False,"Vendor became sanctioned."
    if state.pending_approval:
        if state.pending_approval.action_digest!=digest_action(a):
            return False,"Action changed after approval request."
        status=approval_status(state.pending_approval,a,decisions)
        if status!="APPROVED": return False,f"Approval status: {status}"
    rt,_=route(a,ENVELOPE)
    if rt==Route.DENY: return False,"Current policy denies action."
    return True,"Revalidation passed."

single_decisions=[ApprovalDecision(reviewer="manager:456",approved=True)]
pre_execute(state,single_decisions)

## 11. Hard prohibition beats approval

In [ ]:
SANCTIONED.add("vendor-acme")
print(pre_execute(state,single_decisions))
SANCTIONED.clear()

## 12. Approval expiry

In [ ]:
expired=state.model_copy(deep=True)
expired.pending_approval.expires_at=datetime.now(timezone.utc)-timedelta(seconds=1)
print(pre_execute(expired,single_decisions))

## 13. Revocation / kill switch

In [ ]:
killed=state.model_copy(deep=True)
killed.revoked=True
killed.status="TERMINATED"
print(pre_execute(killed,single_decisions))

## 14. Autonomy budgets

In [ ]:
class BudgetTracker:
    def __init__(self):
        self.spend={}
        self.calls={}
    def can_execute(self,a:Action,e:AutonomyEnvelope):
        spend=self.spend.get(a.task_id,0)+a.amount
        if spend>e.task_spend_limit:
            return False,f"Task spend budget exceeded: projected {spend}"
        return True,"Within aggregate budget."
    def record(self,a:Action):
        self.spend[a.task_id]=self.spend.get(a.task_id,0)+a.amount
        self.calls[a.task_id]=self.calls.get(a.task_id,0)+1

budget=BudgetTracker()
for amt in [3000,3000,3000,3000]:
    x=a.model_copy(update={"amount":amt})
    ok,msg=budget.can_execute(x,ENVELOPE)
    print(amt,ok,msg)
    if ok: budget.record(x)

## 15. Outcome verification

In [ ]:
def verify_outcome(approved_action:Action,result:dict)->dict:
    checks={
      "amount_match":result.get("amount")==approved_action.amount,
      "vendor_match":result.get("vendor_id")==approved_action.vendor_id,
      "status_known":result.get("status") in {"created","settled","cancelled"}
    }
    return {"ok":all(checks.values()),"checks":checks}

print(verify_outcome(high,{"po_id":"PO-1","amount":9000,"vendor_id":"vendor-acme","status":"created"}))
print(verify_outcome(high,{"po_id":"PO-2","amount":19000,"vendor_id":"vendor-acme","status":"created"}))

## 16. Approval-fatigue metrics

In [ ]:
events=pd.DataFrame([
 {"reviewer":"manager:456","risk":.2,"decision":"approve","latency_s":1.0},
 {"reviewer":"manager:456","risk":.3,"decision":"approve","latency_s":.8},
 {"reviewer":"manager:456","risk":.8,"decision":"approve","latency_s":1.1},
 {"reviewer":"manager:456","risk":.9,"decision":"approve","latency_s":.7},
 {"reviewer":"finance:777","risk":.9,"decision":"reject","latency_s":120},
])
summary=events.groupby("reviewer").agg(
    requests=("decision","size"),
    approval_rate=("decision",lambda s:(s=="approve").mean()),
    median_latency_s=("latency_s","median"),
    avg_risk=("risk","mean")
).reset_index()
summary["fatigue_signal"]=(summary.approval_rate>.95)&(summary.median_latency_s<2)
display(summary)

## 17. Adversarial regression suite

In [ ]:
checks=[]

# mutation
s=checkpoint_for_approval(high)
s.action.amount=19000
checks.append(("post-approval mutation",pre_execute(s,single_decisions)[0] is False))

# self approval
req=make_approval(high)
checks.append(("self approval",reviewer_can_decide(req,high,"user:123")[0] is False))

# expired
req2=make_approval(high); req2.expires_at=datetime.now(timezone.utc)-timedelta(seconds=1)
checks.append(("expired approval",reviewer_can_decide(req2,high,"manager:456")[0] is False))

# revoked
s=checkpoint_for_approval(high); s.revoked=True
checks.append(("revoked while waiting",pre_execute(s,single_decisions)[0] is False))

# hard deny
s=checkpoint_for_approval(high); SANCTIONED.add("vendor-acme")
checks.append(("hard deny after approval",pre_execute(s,single_decisions)[0] is False))
SANCTIONED.clear()

df=pd.DataFrame(checks,columns=["test","pass"]); display(df); assert df["pass"].all()

## 18. OpenAI Agents SDK pattern

Current SDK pattern:

```python
from agents import Agent, Runner, function_tool

@function_tool(needs_approval=True)
async def create_po(vendor_id: str, amount: float) -> str:
    ...

agent = Agent(name="Procurement", tools=[create_po])

result = await Runner.run(agent, "Create the PO")

for interruption in result.interruptions:
    # Render exact arguments and enterprise risk context to reviewer.
    state = result.to_state()
    state.approve(interruption)

result = await Runner.run(agent, state)
```

For enterprise use, combine the SDK mechanism with the policy, digest, reviewer-authority, expiry, and revalidation controls implemented above.

## 19. Microsoft Agent Framework pattern

Current framework supports approval-required tools:

```python
from agent_framework import tool

@tool(approval_mode="always_require")
def create_po(vendor_id: str, amount: float) -> str:
    ...
```

Its workflow request/response mechanisms can pause durable workflows for external human input.

Again, the framework supplies the pause/resume primitive. Enterprise governance supplies the approval policy.

# 20. Exercises

### A — Conditional autonomy
Change autonomy level based on anomaly score and incident state.

### B — Two-stage approval
Require manager then finance approval above $25K.

### C — Approval expiry
Use different TTLs by risk tier.

### D — Break glass
Design an emergency override that requires explicit authority and enhanced evidence.

### E — Reviewer workload
Generate 10,000 approval events and tune thresholds to reduce load without increasing missed escalation.

### F — Kill semantics
Model a multi-step workflow and stop new actions after kill while preserving audit state.

### G — Outcome discrepancy
Escalate when executed side effect differs from approved action.

### H — OpenAI Agents SDK
Implement a real conditional `needs_approval` function using the risk router.

### I — Microsoft Agent Framework
Implement the same scenario using approval modes / workflow pause-resume.

### J — Compliance mapping
Map technical controls to EU AI Act Article 14 concepts and ISO/IEC 42001 organizational controls.

# 21. Key takeaways

1. Human presence does not automatically create effective oversight.
2. Treat autonomy as a graduated, governed capability.
3. Define an explicit autonomy envelope.
4. Route human attention according to risk.
5. Approval must show the actual executable action.
6. Bind approval to exact parameters.
7. Revalidate after approval.
8. Hard prohibitions should not disappear because a person clicked approve.
9. Enforce reviewer authority and separation of duties.
10. Use durable state for long-running approvals.
11. Expire approvals and fail safely on timeout.
12. Support pause, redirect, terminate, and revocation.
13. Verify external outcomes.
14. Monitor approval fatigue and oversight quality.
15. Framework HITL features are mechanisms; governance policy still belongs to the enterprise control plane.